# S&P 500 Financial Dashboard
Este notebook extrae métricas financieras clave (ROE, PER, PEG, Crecimiento) para las empresas del S&P 500 y las vuelca en un Google Sheet específico.

In [ ]:
!pip install yfinance gspread

In [ ]:
# ==========================================
#  Screener Profesional: Calidad + Valor (QV Index)
# ==========================================
import pandas as pd
import yfinance as yf
from google.colab import auth
import gspread
from google.auth import default
import requests
import numpy as np

# --- CONFIGURACIÓN ---
SPREADSHEET_NAME = "Trading_Matrix_PremiumSS_Momentum26"
SHEET_NAME = "SP500_Data"  # Se guardará aquí con todos los datos

# 1. OBTENER TICKERS DEL S&P 500
def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    # Add a User-Agent header to mimic a browser request
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        tables = pd.read_html(response.text)
        return tables[0]['Symbol'].tolist()
    except requests.exceptions.RequestException as e:
        print(f"Error obteniendo tickers: {e}")
        return []

# 2. OBTENER DATOS FINANCIEROS (YFINANCE)
def get_financial_data(tickers):
    data = []
    print(f"Descargando datos para {len(tickers)} empresas (esto tomará unos minutos)...")

    for i, ticker in enumerate(tickers):
        yf_ticker = ticker.replace('.', '-')
        try:
            info = yf.Ticker(yf_ticker).info

            # Extracción segura de datos
            data.append({
                'Ticker': ticker,
                'Name': info.get('shortName'),
                'Sector': info.get('sector', 'Unknown'),
                'Price': info.get('currentPrice'),
                'Market Cap (B)': info.get('marketCap', 0) / 1e9 if info.get('marketCap') else None,
                # Métricas de Calidad
                'ROE (%)': info.get('returnOnEquity', 0) * 100 if info.get('returnOnEquity') else None,
                'Profit Margin (%)': info.get('profitMargins', 0) * 100 if info.get('profitMargins') else None,
                'Debt to Equity': info.get('debtToEquity'),
                'Past EPS Growth (%)': info.get('earningsQuarterlyGrowth', 0) * 100 if info.get('earningsQuarterlyGrowth') else None,
                'Expected Growth (%)': info.get('earningsGrowth', 0) * 100 if info.get('earningsGrowth') else None,
                # Métricas de Valoración
                'Forward PER': info.get('forwardPE'),
                'PEG Ratio (5yr)': info.get('trailingPegRatio'), # Proxy común si falta el forward
                'Price/Sales': info.get('priceToSalesTrailing12Months')
            })
        except:
            pass # Si falla uno, seguimos

        if (i+1) % 50 == 0: print(f"Procesados {i+1}...")

    return pd.DataFrame(data)

# 3. CALCULAR ÍNDICE QV (CALIDAD + VALOR)
def calculate_qv_index(df):
    print("Calculando algoritmos de Calidad y Valor...")

    # Columnas numéricas clave
    metrics = ['ROE (%)', 'Profit Margin (%)', 'Debt to Equity', 'Past EPS Growth (%)',
               'Expected Growth (%)', 'Forward PER', 'PEG Ratio (5yr)', 'Price/Sales']

    # Check if DataFrame is empty after filtering for relevant columns. If so, return it as is.
    if df.empty or not any(col in df.columns for col in metrics):
        print("DataFrame is empty or missing key metrics for QV Index calculation. Skipping calculation.")
        return df

    # Limpieza: Convertir a numérico y rellenar nulos con la Mediana del Sector
    for col in metrics:
        if col in df.columns: # Ensure column exists before processing
            df[col] = pd.to_numeric(df[col], errors='coerce')
            # Handle cases where 'Sector' might be missing or all values in a sector are NaN
            # Apply fillna only if there are valid values for median calculation
            df[col] = df.groupby('Sector')[col].transform(lambda x: x.fillna(x.median()) if not x.isnull().all() else x.fillna(df[col].median()))
            df[col] = df[col].fillna(df[col].median()) # Fallback global
        else:
            print(f"Warning: Column '{col}' not found in DataFrame. QV Index calculation might be incomplete.")
            df[col] = np.nan # Add column with NaNs if missing

    # Función de Ranking (0 a 100)
    def rank(df_to_rank, col, asc=True):
        # Ensure the column exists and has non-null values before ranking
        if col in df_to_rank.columns and not df_to_rank[col].isnull().all():
            return df_to_rank.groupby('Sector')[col].rank(pct=True, ascending=asc) * 100
        else:
            return pd.Series([np.nan] * len(df_to_rank), index=df_to_rank.index) # Return NaNs if column is empty or missing

    # --- CALIDAD (Quality) ---
    # Initialize q_score with zeros or NaNs to allow adding scores incrementally
    q_score_components = []
    if 'ROE (%)' in df.columns: q_score_components.append(0.30 * rank(df, 'ROE (%)'))
    if 'Profit Margin (%)' in df.columns: q_score_components.append(0.30 * rank(df, 'Profit Margin (%)'))
    if 'Debt to Equity' in df.columns: q_score_components.append(0.20 * (100 - rank(df, 'Debt to Equity')))
    if 'Past EPS Growth (%)' in df.columns: q_score_components.append(0.10 * rank(df, 'Past EPS Growth (%)'))
    if 'Expected Growth (%)' in df.columns: q_score_components.append(0.10 * rank(df, 'Expected Growth (%)'))

    # Calculate q_score only if components exist, otherwise fill with NaN
    if q_score_components:
        q_score = sum(c.fillna(0) for c in q_score_components) # Fill NaNs in components with 0 before summing
    else:
        q_score = pd.Series([np.nan] * len(df), index=df.index)

    # --- VALORACIÓN (Value) ---
    # Initialize v_score with zeros or NaNs
    v_score_components = []
    if 'PEG Ratio (5yr)' in df.columns: v_score_components.append(0.50 * (100 - rank(df, 'PEG Ratio (5yr)')))
    if 'Forward PER' in df.columns: v_score_components.append(0.30 * (100 - rank(df, 'Forward PER')))
    if 'Price/Sales' in df.columns: v_score_components.append(0.20 * (100 - rank(df, 'Price/Sales')))

    # Calculate v_score only if components exist, otherwise fill with NaN
    if v_score_components:
        v_score = sum(c.fillna(0) for c in v_score_components) # Fill NaNs in components with 0 before summing
    else:
        v_score = pd.Series([np.nan] * len(df), index=df.index)

    # Asignar columnas finales
    df['Quality_Score'] = q_score.round(1)
    df['Value_Score'] = v_score.round(1)
    df['QV_Index'] = (0.6 * q_score + 0.4 * v_score).round(1)

    return df.sort_values(by='QV_Index', ascending=False)

# 4. SUBIR A GOOGLE SHEETS
def upload_to_sheets(df):
    print("Autenticando y subiendo a Google Sheets...")
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    try:
        sh = gc.open(SPREADSHEET_NAME)
        ws = sh.worksheet(SHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        print(f"Spreadsheet '{SPREADSHEET_NAME}' not found. Creating it...")
        sh = gc.create(SPREADSHEET_NAME)
        # Add a default worksheet if the spreadsheet was just created
        ws = sh.add_worksheet(SHEET_NAME, 1000, 20)
    except gspread.exceptions.WorksheetNotFound:
        # If spreadsheet exists but sheet doesn't
        print(f"Sheet '{SHEET_NAME}' not found in '{SPREADSHEET_NAME}'. Creating it...")
        ws = sh.add_worksheet(SHEET_NAME, 1000, 20)

    ws.clear()
    # Reemplazar NaN con string vacío para que Sheets no se queje
    df_clean = df.fillna("")
    # Ensure there are columns to update, otherwise gspread update will fail
    if not df_clean.empty:
        ws.update([df_clean.columns.values.tolist()] + df_clean.values.tolist())
        print(f"¡Éxito! Datos actualizados en {SHEET_NAME}")
    else:
        print(f"DataFrame está vacío. No se actualizaron datos en {SHEET_NAME}")

# --- EJECUCIÓN ---
tickers = get_sp500_tickers()
df_raw = get_financial_data(tickers)
df_final = calculate_qv_index(df_raw)
upload_to_sheets(df_final)

/tmp/ipython-input-2207696664.py:24: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Descargando datos para 503 empresas (esto tomará unos minutos)...
Procesados 50...
Procesados 100...
Procesados 150...
Procesados 200...
Procesados 250...
Procesados 300...
Procesados 350...
Procesados 400...
Procesados 450...
Procesados 500...
Calculando algoritmos de Calidad y Valor...
Autenticando y subiendo a Google Sheets...
¡Éxito! Datos actualizados en SP500_Data


### Resumen de Ticker Específico

Introduce un ticker para ver un resumen de sus métricas financieras clave.

In [ ]:
import yfinance as yf
import pandas as pd

# @param {type:"string"} ticker_symbol
ticker_symbol = 'INCY'

def get_ticker_summary(ticker_symbol):
    yf_ticker = ticker_symbol.replace('.', '-')
    try:
        ticker = yf.Ticker(yf_ticker)
        info = ticker.info

        summary = {
            'Ticker': ticker_symbol,
            'Name': info.get('shortName', 'N/A'),
            'Sector': info.get('sector', 'N/A'),
            'Industry': info.get('industry', 'N/A'),
            'Current Price': info.get('currentPrice', 'N/A'),
            'Market Cap (B)': info.get('marketCap', 0) / 1e9 if info.get('marketCap') else 'N/A',
            'Forward PER': info.get('forwardPE', 'N/A'),
            'PEG Ratio (5yr expected)': info.get('trailingPegRatio', 'N/A'),
            'Price/Sales': info.get('priceToSalesTrailing12Months', 'N/A'),
            'ROE (%)': info.get('returnOnEquity', 0) * 100 if info.get('returnOnEquity') else 'N/A',
            'Profit Margin (%)': info.get('profitMargins', 0) * 100 if info.get('profitMargins') else 'N/A',
            'Debt to Equity': info.get('debtToEquity', 'N/A'),
            'Past EPS Growth (%)': info.get('earningsQuarterlyGrowth', 0) * 100 if info.get('earningsQuarterlyGrowth') else 'N/A',
            'Expected Growth (%)': info.get('earningsGrowth', 0) * 100 if info.get('earningsGrowth') else 'N/A',
            'Website': info.get('website', 'N/A')
        }

        summary_df = pd.DataFrame([summary]).T.rename(columns={0: 'Value'})
        print(f"--- Summary for {ticker_symbol} ---")
        display(summary_df)

    except Exception as e:
        print(f"Error fetching data for {ticker_symbol}: {e}")
        print("Please check the ticker symbol.")

get_ticker_summary(ticker_symbol)


--- Summary for INCY ---


,Value
Ticker,INCY
Name,Incyte Corporation
Sector,Healthcare
Industry,Biotechnology
Current Price,104.67
Market Cap (B),20.549097
Forward PER,13.630716
PEG Ratio (5yr expected),0.1361
Price/Sales,4.269405
ROE (%),30.389
